<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part A: Foundations and Data Exploration</h2>
<h2>Notebook A05: Forecasting Baselines</h2>
</div>

A baseline is a forecast so simple that you would be embarrassed to publish it on its own. That is
exactly what makes it useful: it is the number every serious model has to beat before anyone should care
about it.

Baselines are quick to compute, impossible to overfit, and surprisingly hard to beat on real data. A
gradient boosting model with a validation MAE of 1.9 sounds respectable until you discover that repeating
last year's value scores 1.7. Without that comparison you have no idea whether your model learned
anything at all.

This notebook builds the standard baselines and compares them on a real series.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [Splitting a Time Series](#2.-Splitting-a-Time-Series)
3. [Naive and Seasonal Naive Forecasts](#3.-Naive-and-Seasonal-Naive-Forecasts)
4. [Averaging Methods](#4.-Averaging-Methods)
5. [Comparing the Baselines](#5.-Comparing-the-Baselines)
6. [Comparing Models with AIC and BIC](#6.-Comparing-Models-with-AIC-and-BIC)
7. [Choosing a Baseline](#7.-Choosing-a-Baseline)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

import nb_config

sns.set_theme(style="whitegrid")

We use the CDC regional air temperature dataset, the same one as in Notebooks A01 and A02. If you have
not prepared it yet, run Notebook [F01b](./F01b_Preparing_CDC_dataset.ipynb) first.

Monthly temperature is a good series to learn baselines on, because it has a pattern that some baselines
can exploit and others cannot. That contrast is the whole point of this notebook.

In [ ]:
temperatures = pd.read_parquet(nb_config.CDC_TEMP_PATH)

# One region, at an explicit monthly frequency
series = temperatures["Brandenburg/Berlin"].asfreq("MS")

print(f"Series: {series.name}")
print(f"Range:  {series.index.min().date()} to {series.index.max().date()}")
print(f"Length: {len(series)} months")
series.tail()

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Splitting-a-Time-Series">2. Splitting a Time Series</h3>
</div>

Every forecast needs data it has never seen, otherwise you are measuring memory rather than prediction.

For time series, that split must respect time. The usual `train_test_split(shuffle=True)` from
scikit-learn is wrong here, and not by a small margin: shuffling lets the model train on next January
while being tested on last December. It will look excellent and fail completely in production. Always cut
at a point in time and keep the order.

How much to hold out is a judgement call. A useful rule of thumb is to make the test set at least as long
as the horizon you actually care about forecasting, and long enough to contain a full seasonal cycle so
that seasonal methods can be judged fairly. We hold out the final two years.

In [ ]:
TEST_MONTHS = 24
SEASON_LENGTH = 12  # months in a yearly cycle

train = series.iloc[:-TEST_MONTHS]
test = series.iloc[-TEST_MONTHS:]

print(f"Train: {train.index.min().date()} to {train.index.max().date()}  ({len(train)} months)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()}  ({len(test)} months)")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(train["2015":], color="steelblue", linewidth=1.2, label="Train (from 2015)")
ax.plot(test, color="crimson", linewidth=1.2, label="Test")
ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)

ax.set_title("Train and test split", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Every baseline below is fitted on `train` only, then asked to produce all 24 test values in one go. That
is a **fixed-origin, multi-step** forecast: you stand at the end of the training data and forecast two
years ahead without ever seeing a real value in between.

It is the harder and more honest setting. The alternative, where the model is told the true value after
every step, produces much better-looking numbers and answers a different question. We come back to that
in Notebook [A06](./A06_Evaluating_models.ipynb).

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Naive-and-Seasonal-Naive-Forecasts">3. Naive and Seasonal Naive Forecasts</h3>
</div>

**The naive forecast** predicts that everything from here on equals the last value observed. It sounds
useless, and for a random walk it is provably the best you can do: for many financial series, nothing
beats it.

**The seasonal naive forecast** predicts that each future point equals the value from the same point in
the previous cycle. Next August equals last August. For anything with a strong, stable season, this is
the baseline to beat.

In [ ]:
def naive_forecast(train, horizon):
    """Repeat the last observed value."""
    index = forecast_index(train, horizon)
    return pd.Series(train.iloc[-1], index=index, name="Naive")


def seasonal_naive_forecast(train, horizon, season_length):
    """Repeat the last observed cycle."""
    index = forecast_index(train, horizon)
    last_cycle = train.iloc[-season_length:].to_numpy()
    values = [last_cycle[i % season_length] for i in range(horizon)]
    return pd.Series(values, index=index, name="Seasonal naive")


def forecast_index(train, horizon):
    """The dates the forecast covers, continuing the training index."""
    return pd.date_range(
        start=train.index[-1] + train.index.freq,
        periods=horizon,
        freq=train.index.freq,
    )

In [ ]:
naive = naive_forecast(train, TEST_MONTHS)
seasonal_naive = seasonal_naive_forecast(train, TEST_MONTHS, SEASON_LENGTH)

pd.DataFrame({"Actual": test, "Naive": naive, "Seasonal naive": seasonal_naive}).head(6).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2020":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, label="Actual")
ax.plot(naive, color="crimson", linewidth=1.4, linestyle="--", label="Naive")
ax.plot(seasonal_naive, color="seagreen", linewidth=1.4, linestyle="--", label="Seasonal naive")

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("Naive and seasonal naive forecasts", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The naive forecast is the flat line, and on a seasonal series it is hopeless: it predicts August
temperatures for the following February. The seasonal naive forecast tracks the real series closely,
because German summers and winters repeat far more reliably than they change.

The naive forecast is not a bad method, it is a method whose assumption this series violates. On a series
without seasonality, the ranking would flip.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Averaging-Methods">4. Averaging Methods</h3>
</div>

Averaging methods forecast a constant, but they differ in which observations that constant comes from:

- **Mean forecast.** The average of the entire training set. Assumes the series has a stable level and no
  trend.
- **Simple moving average (SMA).** The average of the last *n* observations. Adapts to a level that
  drifts, at the cost of using less data.
- **Exponential moving average (EMA).** A weighted average where recent observations count for more, with
  the weights decaying geometrically into the past. The `span` controls how fast: a short span reacts
  quickly, a long span is smoother.

All three produce a flat forecast, so like the naive method they cannot capture seasonality. What they
buy you over the naive forecast is robustness: they average out the noise in the final observation.

In [ ]:
def mean_forecast(train, horizon):
    """The average of the whole training series."""
    index = forecast_index(train, horizon)
    return pd.Series(train.mean(), index=index, name="Mean")


def moving_average_forecast(train, horizon, window):
    """The average of the last `window` observations."""
    index = forecast_index(train, horizon)
    return pd.Series(train.iloc[-window:].mean(), index=index, name=f"SMA({window})")


def exponential_moving_average_forecast(train, horizon, span):
    """The exponentially weighted average, giving recent points more weight."""
    index = forecast_index(train, horizon)
    smoothed = train.ewm(span=span, adjust=False).mean().iloc[-1]
    return pd.Series(smoothed, index=index, name=f"EMA({span})")

In [ ]:
mean_fc = mean_forecast(train, TEST_MONTHS)
sma_fc = moving_average_forecast(train, TEST_MONTHS, window=SEASON_LENGTH)
ema_fc = exponential_moving_average_forecast(train, TEST_MONTHS, span=SEASON_LENGTH)

print(f"Mean of the whole series:   {mean_fc.iloc[0]:.2f} °C")
print(f"Mean of the last 12 months: {sma_fc.iloc[0]:.2f} °C")
print(f"EMA with span 12:           {ema_fc.iloc[0]:.2f} °C")

The three constants are much further apart than you might expect, and each gap says something.

The full-series mean is the lowest. That is no coincidence: it averages over 140 years, including a much
cooler nineteenth century, while the moving average only looks at the last twelve months. On a series
with a trend, the window length quietly encodes an assumption about how much history is still relevant.

The EMA is the highest of the three, and by some margin. With a span of 12 it still places about 15% of
its weight on the single most recent observation, and the training data happens to stop in August, at
19.1 °C. On a seasonal series an exponential moving average partly inherits the season of wherever the
data happens to end, which makes it a poor choice of constant here.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2020":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, label="Actual")

for forecast, colour in [(mean_fc, "darkorange"), (sma_fc, "purple"), (ema_fc, "brown")]:
    ax.plot(forecast, color=colour, linewidth=1.4, linestyle="--", label=forecast.name)

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("Averaging baselines", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Three flat lines, none of them close to the shape of the real series. Against a series that swings by
20 °C every year, even a 4 °C spread between the three constants barely matters: all of them are wrong
for most of the year, just in different places.

Note also that the exponential moving average is only being used here to produce a *constant*. Its
smoothed path through the training data is informative, but as a forecast it flattens out immediately.
The models in Part B keep updating that level as they go, which is what makes them more than baselines.

**Exercise.** Build SMA forecasts with windows of 3, 12, and 120 months and plot them together. Which window gives the highest constant, and why? What does that tell you about choosing a window on a series with a long-term trend?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Comparing-the-Baselines">5. Comparing the Baselines</h3>
</div>

Now we can rank them. We use the **mean absolute error**, the average size of the mistakes, in the same
units as the data:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

One metric is enough to compare baselines. Notebook [A06](./A06_Evaluating_models.ipynb) looks properly
at the family of error metrics, what each of them rewards, and how to pick one.

In [ ]:
def mean_absolute_error(actual, forecast):
    """Average absolute difference between actual and forecast values."""
    return float(np.mean(np.abs(actual - forecast)))


forecasts = {
    "Naive": naive,
    "Seasonal naive": seasonal_naive,
    "Mean": mean_fc,
    f"SMA({SEASON_LENGTH})": sma_fc,
    f"EMA({SEASON_LENGTH})": ema_fc,
}

scores = pd.DataFrame(
    [
        {"Baseline": name, "MAE": mean_absolute_error(test.values, forecast.values)}
        for name, forecast in forecasts.items()
    ]
).sort_values("MAE").reset_index(drop=True)

scores.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

colours = ["seagreen" if name == "Seasonal naive" else "steelblue" for name in scores["Baseline"]]
ax.barh(scores["Baseline"], scores["MAE"], color=colours)

for y, value in enumerate(scores["MAE"]):
    ax.text(value + 0.08, y, f"{value:.2f}", va="center", fontsize=10)

ax.invert_yaxis()
ax.set_title("Baseline accuracy (lower is better)", fontsize=14, fontweight="bold")
ax.set_xlabel("MAE (°C)")
ax.set_xlim(0, scores["MAE"].max() * 1.15)
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The seasonal naive forecast wins by a wide margin: its average error is under 2 °C, while every other
baseline is off by 6 °C or more. On a strongly seasonal series that is the expected outcome, and it sets a
demanding target. **Any model you build later has to beat 1.74 °C to have earned its complexity.**

Notice too that the plain naive forecast is the *worst* of the five, well behind even the simple mean. A
baseline is not automatically a weak opponent, and "naive" is not a synonym for "bad". Which baseline is
hard to beat depends entirely on the structure of the series.

**Exercise.** Run the same comparison on a different region, for example `Bayern` or `Schleswig-Holstein`. Does the seasonal naive forecast still win? Try it also on the year-on-year differenced series (`series.diff(12).dropna()`), where the seasonality has been removed. Which baseline wins there?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Comparing-Models-with-AIC-and-BIC">6. Comparing Models with AIC and BIC</h3>
</div>

Forecast accuracy is not the only way to compare models. **Information criteria** score how well a model
fits the data it was trained on, while charging a penalty for every parameter it uses:

- **AIC** (Akaike Information Criterion) balances fit against complexity.
- **BIC** (Bayesian Information Criterion) does the same with a heavier penalty that grows with the
  sample size, so it favours simpler models more strongly.

For both, lower is better, and only the *differences* between models mean anything. The absolute value
depends on the data, so an AIC of 2,253 is neither good nor bad on its own.

Two important limits. First, they are computed from the likelihood of a fitted model, so the baselines
above have no AIC at all: there is no model and nothing to penalise. To demonstrate them we need real
models, so we borrow four exponential smoothing variants from Part B. Second, AIC and BIC can only be
compared between models fitted to *the same data*.

In [ ]:
candidates = {
    "Simple exponential smoothing": SimpleExpSmoothing(train),
    "Holt (trend)": ExponentialSmoothing(train, trend="add"),
    "Seasonal": ExponentialSmoothing(train, seasonal="add", seasonal_periods=SEASON_LENGTH),
    "Holt-Winters (trend + seasonal)": ExponentialSmoothing(
        train, trend="add", seasonal="add", seasonal_periods=SEASON_LENGTH
    ),
}

rows = []
for name, model in candidates.items():
    fitted = model.fit()
    forecast = fitted.forecast(TEST_MONTHS)
    rows.append({
        "Model": name,
        "Parameters": len(fitted.params_formatted),
        "AIC": fitted.aic,
        "BIC": fitted.bic,
        "Test MAE": mean_absolute_error(test.values, forecast.values),
    })

comparison = pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)
comparison.round(1)

Read the table by column. AIC and BIC both rank the **Seasonal** model first, ahead of the fuller
Holt-Winters model that adds a trend component. The extra parameters buy a slightly better fit, but not
enough to pay for themselves.

The last column is the check that matters: on data neither model ever saw, their errors are 1.24 and 1.22
°C. Essentially identical. The information criteria were right that the added trend was not worth it, and
in a case like this they let you reach that conclusion without spending any test data to find out.

Both also comfortably beat the seasonal naive baseline of 1.74 °C, which is what we wanted to see: these
models have earned their complexity. That is the comparison this notebook exists to make possible.

> **A caution.** AIC and BIC measure fit to the training data. A model can win on AIC and still forecast
> badly, particularly if the future does not resemble the past. Use them to narrow a field of candidates,
> never as a substitute for evaluating on held-out data.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Choosing-a-Baseline">7. Choosing a Baseline</h3>
</div>

There is no universally best baseline, but the choice is usually settled by one look at the series:

| If the series... | Use | Why |
|---|---|---|
| has a strong, stable season | **Seasonal naive** | Repeats a pattern that genuinely repeats |
| has no trend and no season | **Mean** | Averages away the noise around a stable level |
| wanders with no clear pattern | **Naive** | Optimal for a random walk; nothing beats the last value |
| has a slowly drifting level | **SMA** or **EMA** | Tracks the level while smoothing short-term noise |

Three habits are worth carrying into the rest of the course:

**Always compute a baseline first.** It costs a few lines and it is the only way to know whether a
complex model is adding anything.

**Report it alongside your model.** "MAE 1.22" means nothing on its own. "MAE 1.22 against a seasonal
naive baseline of 1.74" is a result.

**Pick the baseline that suits the series.** Comparing against a baseline you know is unsuitable, such as
the plain naive forecast here, flatters your model and tells you nothing.

**Exercise.** Take the OPS electricity consumption data (`nb_config.OPS_15M_PATH`), pick one country, and resample it to daily totals. Which of the baselines in this notebook performs best on it, and does the ranking match what you would predict from looking at the series first?

In [ ]:
# Your solution here


---

You now have a reference point for every model in the rest of the course, and a habit of computing it
before anything else.

We used a single metric here to keep the comparison readable. The next notebook takes evaluation
seriously: which error metrics exist, what each one rewards and punishes, how to tell whether a series is
forecastable at all, and why a single train/test split can still mislead you:
[A06 - Evaluating Models](./A06_Evaluating_models.ipynb).

**Solutions.** Worked answers to the 3 exercises above, with the reasoning behind them, are in
[A05_Forecasting_baselines_solutions.ipynb](../solutions/A05_Forecasting_baselines_solutions.ipynb). Try each one yourself first.
